# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tushar-sharma001/Flyrank-Ml-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
%pip -q install duckdb huggingface_hub
import duckdb
import pandas as pd
from google.colab import userdata
from huggingface_hub import HfApi

TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"

api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=TOKEN)

fact_files = [f for f in all_files if "fact_content_daily_performance" in f and f"month={MONTH}" in f]
next_month_files = [f for f in all_files if "fact_content_daily_performance" in f and "month=2026-04" in f]
dim_content_files = [f for f in all_files if "dim_content" in f.lower() and f.endswith(".parquet")]

print("Fact files (March):", fact_files[:5])
print("Fact files (April):", next_month_files[:5])
print("dim_content files:", dim_content_files[:5])

FACT = [f"{BASE}/{f}" for f in fact_files]
FACT_NEXT = [f"{BASE}/{f}" for f in next_month_files]
DIM_CONTENT = [f"{BASE}/{f}" for f in dim_content_files]

FACT_STR = "[" + ", ".join(f"'{p}'" for p in FACT) + "]"
FACT_TWO_MONTHS_STR = "[" + ", ".join(f"'{p}'" for p in FACT + FACT_NEXT) + "]"
DIM_CONTENT_STR = "[" + ", ".join(f"'{p}'" for p in DIM_CONTENT) + "]"

# Schema checks up front — no more one-error-at-a-time guessing
print("\n--- dim_content columns ---")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet({DIM_CONTENT_STR})").df()["column_name"].tolist())
print("\n--- fact table columns ---")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet({FACT_STR})").df()["column_name"].tolist())

Fact files (March): ['fact_content_daily_performance/month=2026-03/data_0.parquet']
Fact files (April): ['fact_content_daily_performance/month=2026-04/data_0.parquet']
dim_content files: ['dim_content.parquet']

--- dim_content columns ---
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

--- fact table columns ---
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_enga

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** a page is worth reviewing for refresh if it hasn't been
touched in a long time (stale) AND it's still getting real search visibility (visible).

**Two signals checked before trusting this rule:**
1. Staleness (behind FlyRank's refresh flags) — does time-since-update relate to future decline?
2. CTR-vs-position (behind the CTR-fix logic) — does CTR actually drop as position worsens?

Reason code: `stale_but_visible`. Actions: `review_for_refresh` / `no_action_needed`.

In [12]:
df_content = con.sql(f"SELECT content_hash_id, content_created_date FROM read_parquet({DIM_CONTENT_STR})").df()

labeled = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, gsc_impressions,
           LEAD(gsc_impressions, 30) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS impressions_plus30
    FROM read_parquet({FACT_TWO_MONTHS_STR})
""").df()
labeled = labeled[labeled["report_date"] < "2026-04-01"].dropna(subset=["impressions_plus30"])
labeled["future_decline_label"] = (labeled["impressions_plus30"] < labeled["gsc_impressions"]).astype(int)

labeled = labeled.merge(df_content, on="content_hash_id", how="left")
labeled["report_date"] = pd.to_datetime(labeled["report_date"])
labeled["content_created_date"] = pd.to_datetime(labeled["content_created_date"])
labeled["days_since_update"] = (labeled["report_date"] - labeled["content_created_date"]).dt.days

bins = [0, 90, 180, 365, 100000]
tier_labels = ["<90d", "90-180d", "180-365d", "365d+"]
labeled["staleness_tier"] = pd.cut(labeled["days_since_update"], bins=bins, labels=tier_labels)

staleness_table = labeled.groupby("staleness_tier", observed=True).agg(
    n=("future_decline_label", "size"),
    decline_rate=("future_decline_label", "mean")
).reset_index()
print(staleness_table)
print("\nVerdict: [fill in CONFIRMED / OPPOSITE / MIXED / FALSE once you see the real table]")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  staleness_tier        n  decline_rate
0           <90d  2225193      0.325105
1        90-180d  1430290      0.282985
2       180-365d  5280077      0.164539
3          365d+   808840      0.306525

Verdict: [fill in CONFIRMED / OPPOSITE / MIXED / FALSE once you see the real table]


In [13]:
ctr_check = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1-3'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            ELSE '20+'
        END AS position_tier,
        COUNT(*) AS n,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END) AS avg_ctr
    FROM read_parquet({FACT_STR})
    WHERE gsc_impressions > 0
    GROUP BY 1
    ORDER BY 1
""").df()
print(ctr_check)
print("\nVerdict: [fill in once you see whether CTR actually drops as position tier worsens]")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_tier        n   avg_ctr
0           1-3   727362  0.004756
1         11-20   519223  0.002770
2           20+   908354  0.001289
3          4-10  1456122  0.003473

Verdict: [fill in once you see whether CTR actually drops as position tier worsens]


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score, reason code, action label — no future-window or label-derived inputs.

In [14]:
features = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_prior90,
           MAX(gsc_avg_position) AS avg_position
    FROM read_parquet({FACT_STR})
    GROUP BY content_hash_id, client_hash_id
""").df()
features = features.merge(df_content, on="content_hash_id", how="left")
features["content_created_date"] = pd.to_datetime(features["content_created_date"])
features["days_since_update"] = (pd.Timestamp("2026-03-31") - features["content_created_date"]).dt.days

stale = (features["days_since_update"] >= 180).astype(int)
visible = (features["impressions_prior90"] >= 500).astype(int)
features["score"] = stale * visible * features["impressions_prior90"]
features["reason_code"] = "stale_but_visible"
features["action"] = features["score"].apply(lambda s: "review_for_refresh" if s > 0 else "no_action_needed")

ranked = features.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(ranked)} rows. Non-zero scores: {(ranked['score'] > 0).sum()}")
ranked.head(10)

Wrote 331437 rows. Non-zero scores: 31464


,content_hash_id,client_hash_id,impressions_prior90,avg_position,content_created_date,days_since_update,score,reason_code,action
0,content_eadb33b5df496f4a,client_e547b89c05043229,617124.0,2.768246,2025-03-21,375,617124.0,stale_but_visible,review_for_refresh
1,content_ec2e0346994fb5a5,client_e547b89c05043229,245276.0,3.768936,2025-01-21,434,245276.0,stale_but_visible,review_for_refresh
2,content_e8a52cf3d5988c07,client_23a62021009f63c4,244931.0,18.481879,2025-08-13,230,244931.0,stale_but_visible,review_for_refresh
3,content_0e03de7680314cd5,client_e547b89c05043229,221310.0,3.325857,2025-03-21,375,221310.0,stale_but_visible,review_for_refresh
4,content_e7b5dd4dff461ad2,client_08a6a72ff48e62c0,205045.0,5.594397,2025-04-22,343,205045.0,stale_but_visible,review_for_refresh
5,content_8d7d99f109e19aa2,client_e547b89c05043229,203497.0,3.169343,2025-03-21,375,203497.0,stale_but_visible,review_for_refresh
6,content_36e53e9c707674fc,client_23a62021009f63c4,194579.0,35.693847,2025-08-14,229,194579.0,stale_but_visible,review_for_refresh
7,content_4ffe18112a5642e3,client_e547b89c05043229,186983.0,2.522659,2025-03-21,375,186983.0,stale_but_visible,review_for_refresh
8,content_471d9cabce329a66,client_73cda7b4e4f265ea,164885.0,6.188278,2025-03-21,375,164885.0,stale_but_visible,review_for_refresh
9,content_512dbad65bd5ade9,client_73cda7b4e4f265ea,154358.0,3.430564,2025-09-25,187,154358.0,stale_but_visible,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

One line each: the action, why it's there, what would make it wrong.

In [15]:
ranked[["content_hash_id", "client_hash_id", "score", "reason_code", "action",
        "impressions_prior90", "avg_position", "days_since_update"]].head(10)

,content_hash_id,client_hash_id,score,reason_code,action,impressions_prior90,avg_position,days_since_update
0,content_eadb33b5df496f4a,client_e547b89c05043229,617124.0,stale_but_visible,review_for_refresh,617124.0,2.768246,375
1,content_ec2e0346994fb5a5,client_e547b89c05043229,245276.0,stale_but_visible,review_for_refresh,245276.0,3.768936,434
2,content_e8a52cf3d5988c07,client_23a62021009f63c4,244931.0,stale_but_visible,review_for_refresh,244931.0,18.481879,230
3,content_0e03de7680314cd5,client_e547b89c05043229,221310.0,stale_but_visible,review_for_refresh,221310.0,3.325857,375
4,content_e7b5dd4dff461ad2,client_08a6a72ff48e62c0,205045.0,stale_but_visible,review_for_refresh,205045.0,5.594397,343
5,content_8d7d99f109e19aa2,client_e547b89c05043229,203497.0,stale_but_visible,review_for_refresh,203497.0,3.169343,375
6,content_36e53e9c707674fc,client_23a62021009f63c4,194579.0,stale_but_visible,review_for_refresh,194579.0,35.693847,229
7,content_4ffe18112a5642e3,client_e547b89c05043229,186983.0,stale_but_visible,review_for_refresh,186983.0,2.522659,375
8,content_471d9cabce329a66,client_73cda7b4e4f265ea,164885.0,stale_but_visible,review_for_refresh,164885.0,6.188278,375
9,content_512dbad65bd5ade9,client_73cda7b4e4f265ea,154358.0,stale_but_visible,review_for_refresh,154358.0,3.430564,187


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Which top-10 picks look weakest, and why? Confirming no product flags or future windows leaked in.

In [16]:
used_columns = ["impressions_prior90", "avg_position", "days_since_update"]
banned = ["health_score", "priority_score", "action_type", "future_decline_label", "impressions_plus30"]
leaked = [c for c in used_columns if c in banned]
print(f"Leakage check — banned columns used in the score: {leaked}")
print("Empty list confirms the rule uses only prior-window, non-product-flag inputs.")

Leakage check — banned columns used in the score: []
Empty list confirms the rule uses only prior-window, non-product-flag inputs.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.